In [149]:
from sklearn.linear_model import Ridge
from dataclasses import dataclass
from datetime import datetime
from fredapi import Fred
import yfinance as yf
import pandas as pd
import numpy as np
import os

In [148]:
class DataStore:
  def __init__(self, debug:bool=False):
    self.debug = debug

  def _get_data(
      self,
      tickers:list,
      start:str,
      end:str,
      interval:str="1d"
  ):
    tmp = []
    for t in tickers:
      if isinstance(t, list):
        tmp.extend(t)
      elif isinstance(t, str):
        tmp.append(t)

      else:
        print(f"Warning: Skipping {t} | type: {type(t)} ")

    tickers_clean = list(set(tmp))

    df_path = f"portfolio_{start}_{end}.parquet"

    if not os.path.exists(df_path):
      df = yf.download(tickers_clean, start, end, interval)["Close"]
      df.to_parquet(df_path)

    else:
      df = pd.read_parquet(df_path)

    returns = df.pct_change()

    return returns

  def _get_excess_returns(self, benchmark:str, start:str, end:str):
    r = yf.download([benchmark, "^IRX"], start, end)["Close"]
    bench_r = r[benchmark].pct_change()
    rf_annual =  r["^IRX"] / 100
    rf_daily = rf_annual / 252
    ex_r = bench_r - rf_daily

    return pd.DataFrame(ex_r, columns=[benchmark])


  def get_factors(self, start:str, end:str, fred_api_key:str, benchmark:str="^GSPC"):
    factors = []
    fred_df = self._fetch_fred(fred_api_key)
    factors.append(fred_df)

    equity_ex_r_df = self._get_excess_returns(benchmark, start, end)
    factors.append(equity_ex_r_df)

    pd.concat(factors, axis=1, join='outer')
    combined_df = combined_df.sort_index()
    cropped_df = combined_df.loc[start:end]

    return cropped_df

  def _get_breakeven_inf(self, fred):
    nominal_yield = fred.get_series("DGS5")
    real_yield = fred.get_series("DFII5")
    breakeven = nominal_yield - real_yield

    return pd.DataFrame(breakeven, columns=["BreakevenInf"])

  def _fetch_fred(self, fred_api_key):
    fred = Fred(api_key=fred_api_key)

    nfci_series = fred.get_series("NFCI")
    usd_series = fred.get_series('DTWEXBGS')
    tips_10y = fred.get_series('DFII10')
    oas = fred.get_series('BAMLH0A0HYM2')

    oas = pd.DataFrame(tips_10y, columns=['HighYieldOAS'])
    tips = pd.DataFrame(tips_10y, columns=['10YTIPSYield'])
    nfci = pd.DataFrame(nfci_series, columns=["NFCIValue"])
    usd = pd.DataFrame(usd_series, columns=["DTWEXBGS"])
    inf = self._get_breakeven_inf(fred)

    fred_df = pd.concat(
      [nfci, inf, usd, oas, tips],
      axis=1,
      join='outer'
    ).pct_change().dropna()

    return fred_df




In [132]:
ds = DataStore()

In [133]:
returns = ds._get_data(
  tickers=tickers,
  start="2020-01-01",
  end="2027-01-01",
)

In [113]:
@dataclass
class FilterParams:
  min_length:tuple = (5, "y")

  def __post_init__(self):
    if not self.min_length[1] in ["d", "m", "y"]:
      raise ValueError(f"Invalid time period: {self.min_length[1]}")

    if self.min_length[0] <= 0:
      raise ValueError(f"Invalid min length: {self.min_length[0]}")

In [ ]:
class Filter:
  def __init__(self, filter_params:FilterParams, debug:bool=True):
    self.debug = debug
    self.p = filter_params

  def _time_horizon_filter(self, returns:pd.DataFrame) -> pd.DataFrame:
    end:datetime = pd.to_datetime(
      returns.index.max()
    )

    min_l = self.p.min_length[0]
    tp = self.p.min_length[1]

    min_start:datetime = (
        end
        - pd.DateOffset(
          years = min_l if tp == "y" else 0,
          months = min_l if tp == "m" else 0,
          days = min_l if tp == "d" else 0,
        )
    )
    if self.debug:
      print(f"Min start: {min_start}")

    drop = []

    for t in returns.columns:
      min_t = pd.to_datetime(returns[t].dropna().index.min())

      if min_t > min_start:
        drop.append(t)

    returns = returns.drop(columns=drop).dropna()

    return returns

def _fit_ridge(self, X, y):
  model = Ridge(alpha=1.0)
  model.fit(X, y)

  betas = model.coef_
  beta_table = pd.DataFrame({
    'Feature': X.columns,
    'Beta (Coefficient)': betas
  })

  return beta_table

def _factor_exposure_filter(self, returns, factors):
  drop = []
  factor_exposures = {}

  for i in returns.columns:
    factor_exposures[i] = self._fit_ridge(factors, returns[i])

  factor_exposures_df = pd.concat(factor_exposures, axis=1)
  factor_exposures_df.columns.names = ['Ticker', 'Factor']

  return returns.drop(columns=drop)




